# prime-rl S5: SatelliteAgent toy RL (RTX 6000 Pro offline)

End-to-end smoke of SatelliteAgent's `satelliteagent_env` (verifiers
StatefulToolEnv wrapping submit_to_ground/drop tools) under prime-rl's RL
loop. Uses the toy submit-or-drop dataset bundled in `satelliteagent_env`
(10 hand-crafted scenarios) so we can verify:

- Tool calls actually fire (orchestrator -> vLLM -> ToolEnv -> python)
- Reward (`action_match`) flows back to the trainer
- 3-process pipeline runs to completion with our env (not just reverse-text)

Inputs (kernel_sources):
- `prime-rl-offline-prep` (wheels + Qwen3-0.6B + HF cache)
- `satelliteagent-env-prep` (satellite-agent wheel exposing `satelliteagent_env`)

Tiny config: `max_steps=2 batch_size=4 rollouts_per_example=2 seq_len=512`.

## 1. Install (S3 recipe + satellite-agent wheel)

In [ ]:
import os, subprocess, glob, re

PREP_MAIN = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
PREP_ENV  = "/kaggle/input/notebooks/<KAGGLE_USER>/satelliteagent-env-prep"
BASE_MAIN = f"{PREP_MAIN}/output" if os.path.exists(f"{PREP_MAIN}/output") else PREP_MAIN
BASE_ENV  = f"{PREP_ENV}/output"  if os.path.exists(f"{PREP_ENV}/output")  else PREP_ENV
WHEELS_MAIN = f"{BASE_MAIN}/wheels"
WHEELS_ENV  = f"{BASE_ENV}/wheels"
MODEL_DIR = f"{BASE_MAIN}/models/Qwen3-0.6B"

print(f"PREP_MAIN wheels: {WHEELS_MAIN}")
print(f"PREP_ENV  wheels: {WHEELS_ENV}")
subprocess.run(f"ls {WHEELS_ENV} 2>&1 | head -5", shell=True)

# Skip GPU stack + science stack (Kaggle preinstalled)
SKIP_PREFIXES = (
    "torch-", "torchvision-", "torchaudio-", "nvidia_",
    "numpy-", "scipy-", "scikit_learn-", "pandas-",
)

def parse_version(name):
    m = re.match(r"([a-zA-Z0-9_]+?)-(\d+(?:\.\d+)*)", os.path.basename(name))
    if not m:
        return None, None
    return m.group(1).lower(), tuple(int(x) for x in m.group(2).split("."))

all_wheels = sorted(glob.glob(f"{WHEELS_MAIN}/*.whl") + glob.glob(f"{WHEELS_ENV}/*.whl"))
remaining = [w for w in all_wheels if not os.path.basename(w).startswith(SKIP_PREFIXES)]
keep = {}
for w in remaining:
    pkg, ver = parse_version(w)
    if pkg is None:
        keep[os.path.basename(w)] = (None, w); continue
    if pkg in keep and keep[pkg][0] is not None:
        if ver > keep[pkg][0]:
            keep[pkg] = (ver, w)
    else:
        keep[pkg] = (ver, w)
filtered = [v[1] for v in keep.values()]
print(f"installing {len(filtered)} wheels")
subprocess.run(
    "python3.12 -m pip install --no-index --no-build-isolation --pre --no-deps "
    + " ".join(filtered),
    shell=True, check=True,
)
print()
subprocess.run("python3.12 -c 'import satelliteagent_env; print(\"satelliteagent_env OK,\", satelliteagent_env.load_environment)'", shell=True, check=True)

## 2. Write split TOMLs (trainer / inference / orchestrator)

In [ ]:
import os, subprocess

PREP_MAIN = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
BASE_MAIN = f"{PREP_MAIN}/output" if os.path.exists(f"{PREP_MAIN}/output") else PREP_MAIN
MODEL_DIR = f"{BASE_MAIN}/models/Qwen3-0.6B"

TRAINER_TOML = f'''max_steps = 2

[model]
name = "{MODEL_DIR}"
seq_len = 512

[optim]
lr = 3e-6

[ckpt]
'''

# Qwen3 uses Hermes-style tool calls; explicitly set the parser so vLLM
# enables auto tool choice (otherwise we get "auto tool choice requires
# --enable-auto-tool-choice and --tool-call-parser to be set").
INFER_TOML = f'''gpu_memory_utilization = 0.5

[model]
name = "{MODEL_DIR}"
max_model_len = 512
enforce_eager = true
tool_call_parser = "hermes"

[server]
port = 8000
'''

# satelliteagent_env: toy mode (10 scenarios, submit_to_ground vs drop).
# filters = [] because base Qwen3-0.6B may produce zero-variance reward groups.
ORCH_TOML = f'''max_steps = 2
batch_size = 4
seq_len = 512
rollouts_per_example = 2
filters = []

[model]
name = "{MODEL_DIR}"

[train.sampling]
max_completion_tokens = 128

[[train.env]]
id = "satelliteagent_env"
'''

os.makedirs("/kaggle/working", exist_ok=True)
with open("/kaggle/working/trainer.toml", "w") as f: f.write(TRAINER_TOML)
with open("/kaggle/working/infer.toml",   "w") as f: f.write(INFER_TOML)
with open("/kaggle/working/orch.toml",    "w") as f: f.write(ORCH_TOML)
for p in ["/kaggle/working/trainer.toml", "/kaggle/working/infer.toml", "/kaggle/working/orch.toml"]:
    print(f"=== {p} ===")
    subprocess.run(f"cat {p}", shell=True)
    print()

## 3. Run 3-process RL (vLLM + orchestrator + trainer)

In [ ]:
import os, subprocess, time, urllib.request

LOG_DIR = "/kaggle/working/proc_logs"
os.makedirs(LOG_DIR, exist_ok=True)
API_KEY = "dummy"

PREP_MAIN = "/kaggle/input/notebooks/<KAGGLE_USER>/prime-rl-offline-prep"
BASE_MAIN = f"{PREP_MAIN}/output" if os.path.exists(f"{PREP_MAIN}/output") else PREP_MAIN
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)

env = os.environ.copy()
env.update({
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "WANDB_MODE": "disabled",
    "CUDA_VISIBLE_DEVICES": "0",
    "VLLM_API_KEY": API_KEY,
    "HF_HOME": f"{BASE_MAIN}/hf_cache",
    "HF_HUB_CACHE": f"{BASE_MAIN}/hf_cache",
    "HF_DATASETS_CACHE": "/kaggle/working/hf_cache",
})

procs = {}
def start_bg(name, cmd):
    log_path = f"{LOG_DIR}/{name}.log"
    log_f = open(log_path, "wb")
    p = subprocess.Popen(cmd, shell=True, env=env, stdout=log_f, stderr=subprocess.STDOUT)
    procs[name] = (p, log_f, log_path)
    print(f"[{name}] started pid={p.pid}")
    return p

def cleanup():
    for name, (p, f, _) in procs.items():
        if p.poll() is None:
            try: p.terminate(); p.wait(timeout=10)
            except: p.kill()
        f.close()

def tail(path, n=40):
    try: return subprocess.check_output(f"tail -n {n} {path}", shell=True).decode(errors="replace")
    except Exception as e: return f"<tail error: {e}>"

try:
    start_bg("inference", "python3.12 -m prime_rl.entrypoints.inference @ /kaggle/working/infer.toml")
    print("\nWaiting for vLLM ...")
    ready = False; deadline = time.time() + 600
    while time.time() < deadline:
        if procs["inference"][0].poll() is not None:
            raise RuntimeError(f"inference died early. Tail:\n{tail(procs['inference'][2], 80)}")
        try:
            req = urllib.request.Request("http://localhost:8000/v1/models", headers={"Authorization": f"Bearer {API_KEY}"})
            if urllib.request.urlopen(req, timeout=5).status == 200:
                ready = True; break
        except: pass
        time.sleep(5)
    if not ready:
        raise RuntimeError(f"inference not ready in 10min. Tail:\n{tail(procs['inference'][2], 80)}")
    print("inference is up.")

    start_bg("orchestrator", "python3.12 -m prime_rl.orchestrator.orchestrator @ /kaggle/working/orch.toml")
    time.sleep(5)
    if procs["orchestrator"][0].poll() is not None:
        raise RuntimeError(f"orchestrator died early. Tail:\n{tail(procs['orchestrator'][2], 80)}")
    print("orchestrator is up.")

    print("\nStarting trainer (live via tee)...")
    print("=" * 60)
    trainer_log = f"{LOG_DIR}/trainer.log"
    t0 = time.time()
    trainer_cmd = (
        "set -o pipefail; "
        "python3.12 -m torch.distributed.run --nproc-per-node 1 "
        "-m prime_rl.trainer.rl.train @ /kaggle/working/trainer.toml "
        f"2>&1 | tee {trainer_log}"
    )
    trainer = subprocess.run(["bash", "-c", trainer_cmd], env=env)
    elapsed = time.time() - t0
    print("=" * 60)
    print(f"\ntrainer exit={trainer.returncode}  elapsed={elapsed:.1f}s")
    if trainer.returncode != 0:
        print("\n--- inference tail ---");    print(tail(procs["inference"][2], 40))
        print("\n--- orchestrator tail ---"); print(tail(procs["orchestrator"][2], 40))
        raise RuntimeError(f"trainer failed exit={trainer.returncode}")
    print("\nS5 PASS")
finally:
    cleanup()